In [ ]:
import random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
print('Libraries loaded')

In [ ]:
SR       = 32000
DURATION = 5
N_MFCC   = 20
SEED     = 42

BASE_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE_DIR / 'train_audio'
TRAIN_SC    = BASE_DIR / 'train_soundscapes'
META_CSV    = BASE_DIR / 'train.csv'
TAX_CSV     = BASE_DIR / 'taxonomy.csv'
SC_CSV      = BASE_DIR / 'train_soundscapes_labels.csv'

FEAT_CACHE  = Path('/kaggle/working/train_features.npz')

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
df  = pd.read_csv(META_CSV)
tax = pd.read_csv(TAX_CSV)
sc  = pd.read_csv(SC_CSV)

species_list   = tax['primary_label'].astype(str).tolist()
species_to_idx = {sp: i for i, sp in enumerate(species_list)}
NUM_CLASSES    = len(species_list)

counts   = df['primary_label'].astype(str).value_counts()
valid_df = df[df['primary_label'].astype(str).isin(counts[counts >= 2].index)].reset_index(drop=True)

print(f'Species to predict : {NUM_CLASSES}')
print(f'Training clips     : {len(valid_df)}')
print(f'Soundscape windows : {len(sc)}')

## Feature Extraction

Per 5-second window: 20 MFCCs (mean + std) + spectral centroid + ZCR = **42 features**.

In [ ]:
def extract_features(audio):
    target = SR * DURATION
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    audio = audio[:target]

    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC)
    cent = librosa.feature.spectral_centroid(y=audio, sr=SR)
    zcr  = librosa.feature.zero_crossing_rate(y=audio)

    return np.concatenate([
        mfcc.mean(axis=1),   # 20 values
        mfcc.std(axis=1),    # 20 values
        cent.mean(axis=1),   #  1 value
        zcr.mean(axis=1),    #  1 value
    ]).astype(np.float32)

In [ ]:
if FEAT_CACHE.exists():
    print('Loading cached features...')
    data    = np.load(FEAT_CACHE)
    X_train = data['X']
    y_train = data['y']
    print(f'Loaded {len(X_train)} clips from cache')
else:
    print(f'Extracting features from {len(valid_df)} clips (first run only)...')
    t0 = time.time()
    X_rows, y_rows = [], []

    for i, (_, row) in enumerate(valid_df.iterrows()):
        fp = TRAIN_AUDIO / row['filename']
        try:
            total    = librosa.get_duration(path=str(fp))
            offset   = max(0.0, (total - DURATION) / 2.0)
            audio, _ = librosa.load(str(fp), sr=SR, offset=offset, duration=DURATION)
            X_rows.append(extract_features(audio))
            y_rows.append(species_to_idx.get(str(row['primary_label']), -1))
        except Exception:
            continue
        if (i + 1) % 2000 == 0:
            print(f'  {i + 1}/{len(valid_df)}  ({time.time() - t0:.0f}s elapsed)')

    X_train = np.array(X_rows, dtype=np.float32)
    y_train = np.array(y_rows, dtype=np.int32)
    mask    = y_train >= 0
    X_train, y_train = X_train[mask], y_train[mask]

    np.savez(FEAT_CACHE, X=X_train, y=y_train)
    print(f'Saved to cache. {len(X_train)} clips in {time.time() - t0:.0f}s')

print(f'Feature matrix : {X_train.shape}  ({X_train.shape[1]} features, {len(set(y_train))} classes)')

## Classifier Comparison

Three classifiers trained on identical features, evaluated on 20% held-out soundscape windows.

In [ ]:
# ── Soundscape validation set (20% of labelled windows) ─────────────────────
sc_val = sc.sample(frac=0.2, random_state=SEED)
sc_val_X, sc_val_Y = [], []

for _, row in sc_val.iterrows():
    parts     = str(row['start']).split(':')
    start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    try:
        audio, _ = librosa.load(str(TRAIN_SC / row['filename']),
                                sr=SR, offset=float(start_sec), duration=DURATION)
        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        for sp in str(row['primary_label']).split(';'):
            if sp.strip() in species_to_idx:
                target[species_to_idx[sp.strip()]] = 1.0
        sc_val_X.append(extract_features(audio))
        sc_val_Y.append(target)
    except Exception:
        continue

if not sc_val_X:
    raise RuntimeError('No soundscape validation samples loaded — check TRAIN_SC path')

X_val = np.array(sc_val_X, dtype=np.float32)
Y_val = np.array(sc_val_Y, dtype=np.float32)
print(f'Validation set : {len(X_val)} soundscape windows')


# ── Evaluation helper ────────────────────────────────────────────────────────
def macro_auc(clf, X, Y):
    P_partial = clf.predict_proba(X)                     # (n, seen_classes)
    P = np.zeros((len(X), NUM_CLASSES), dtype=np.float32)
    for col_i, class_i in enumerate(clf.classes_):
        P[:, class_i] = P_partial[:, col_i]             # expand to full 234-class space
    aucs = [
        roc_auc_score(Y[:, i], P[:, i])
        for i in range(NUM_CLASSES)
        if 0 < Y[:, i].sum() < len(Y)
    ]
    return round(float(np.mean(aucs)), 4)


# ── Classifiers ──────────────────────────────────────────────────────────────
classifiers = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=1000, solver='saga',
                                      n_jobs=-1, random_state=SEED)),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, n_jobs=-1, random_state=SEED),
    'Linear SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    CalibratedClassifierCV(LinearSVC(max_iter=1000, random_state=SEED))),
    ]),
}

# ── Train + evaluate each ────────────────────────────────────────────────────
results = {}
for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    auc     = macro_auc(clf, X_val, Y_val)
    elapsed = round(time.time() - t0, 1)
    results[name] = {'Val AUC (macro)': auc, 'Train time (s)': elapsed}
    print(f'{name:22s}  AUC = {auc:.4f}  ({elapsed}s)')

# ── Comparison matrix ────────────────────────────────────────────────────────
print()
print(pd.DataFrame(results).T.to_string())